# Étape 1 – Exploration du jeu de données D-Fire

**Projet ForestFireDetection** – Détection précoce de feux de forêt (fumée / feu)
Objectif : charger le dataset D-Fire, vérifier la qualité des images et des annotations,
décrire la distribution des classes et des sources, puis produire un fichier de métadonnées
réutilisé par les étapes suivantes (préparation, clustering, semi-supervisé).

**Source** : [D-Fire](https://github.com/gaia-solutions-on-demand/DFireDataset) – licence CC0 1.0
(les images `PublicDataset*` restent soumises aux conditions de leurs fournisseurs d'origine).
Annotations au format YOLO : `<classe> <x_centre> <y_centre> <largeur> <hauteur>`, coordonnées normalisées dans [0, 1].

## 1. Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image  # Lecture et inspection des fichiers image

## 2. Chemins du dataset

In [ ]:
# Racine du projet : le notebook peut être lancé depuis notebooks/ ou depuis la racine
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data"
DFIRE_DIR = DATA_DIR / "D-Fire"          # Dataset brut (non versionné)
PROCESSED_DIR = DATA_DIR / "processed"   # Fichiers produits par les notebooks
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "test"]               # Découpage officiel fourni par D-Fire
CLASS_NAMES = {0: "smoke", 1: "fire"}    # Convention D-Fire (vérifiée en section 5)
RANDOM_STATE = 42                        # Graine pour la reproductibilité

# Vérification rapide : nombre d'images et de labels par split
for split in SPLITS:
    n_img = len(list((DFIRE_DIR / split / "images").glob("*")))
    n_lbl = len(list((DFIRE_DIR / split / "labels").glob("*.txt")))
    print(f"{split:5s} : {n_img:6d} images | {n_lbl:6d} fichiers de labels")

## 3. Lecture des annotations

Chaque fichier `.txt` contient une ligne par objet. Un fichier **vide** = image sans feu ni fumée
(image négative), indispensable pour limiter les fausses alertes.

In [ ]:
def parse_label_file(path):
    """Lit un fichier YOLO -> (liste de boîtes, nb de lignes invalides)."""
    boxes, n_invalid = [], 0
    for line in path.read_text().splitlines():
        parts = line.split()
        if not parts:            # ligne vide : ignorée
            continue
        if len(parts) != 5:      # format attendu : 5 champs
            n_invalid += 1
            continue
        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except ValueError:       # champ non numérique
            n_invalid += 1
            continue
        boxes.append((cls, x, y, w, h))
    return boxes, n_invalid


def image_category(classes):
    """Catégorie au niveau image à partir des classes présentes."""
    has_smoke, has_fire = 0 in classes, 1 in classes
    if has_smoke and has_fire:
        return "fire_and_smoke"
    if has_fire:
        return "fire_only"
    if has_smoke:
        return "smoke_only"
    return "none"


def image_source(stem):
    """Source déduite du préfixe du nom de fichier (AoF, WEB, PublicDataset)."""
    match = re.match(r"[A-Za-z]+", stem)
    return match.group(0) if match else "inconnue"

## 4. Construction des DataFrames de métadonnées

Deux tables :
- `df` : **une ligne par image** (split, source, catégorie, nombre de boîtes…)
- `df_boxes` : **une ligne par boîte** (classe, position, taille)

Les chemins sont stockés **relativement à la racine du projet** pour que le CSV reste
utilisable sur une autre machine.

In [ ]:
records, box_records = [], []

for split in SPLITS:
    images_dir = DFIRE_DIR / split / "images"
    labels_dir = DFIRE_DIR / split / "labels"
    for img_path in sorted(images_dir.glob("*.jpg")):
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        has_label_file = lbl_path.exists()
        boxes, n_invalid = parse_label_file(lbl_path) if has_label_file else ([], 0)
        classes = {b[0] for b in boxes}

        records.append({
            "image_id": img_path.stem,
            "path": img_path.relative_to(ROOT_DIR).as_posix(),
            "split": split,
            "source": image_source(img_path.stem),
            "has_label_file": has_label_file,
            "n_invalid_lines": n_invalid,
            "n_boxes": len(boxes),
            "n_smoke": sum(b[0] == 0 for b in boxes),
            "n_fire": sum(b[0] == 1 for b in boxes),
            "has_smoke": int(0 in classes),
            "has_fire": int(1 in classes),
            "category": image_category(classes),
        })
        for cls, x, y, w, h in boxes:
            box_records.append({
                "image_id": img_path.stem, "split": split,
                "source": image_source(img_path.stem),
                "class_id": cls, "x": x, "y": y, "w": w, "h": h,
            })

df = pd.DataFrame(records)
df_boxes = pd.DataFrame(box_records)
df_boxes["class_name"] = df_boxes["class_id"].map(CLASS_NAMES)
df_boxes["area"] = df_boxes["w"] * df_boxes["h"]   # aire relative à l'image (0–1)

print(f"df       : {df.shape}")
print(f"df_boxes : {df_boxes.shape}")
df.head()

## 5. Contrôles de qualité des annotations

In [ ]:
# Fichiers orphelins : labels sans image correspondante
orphan_labels = []
for split in SPLITS:
    image_stems = {p.stem for p in (DFIRE_DIR / split / "images").glob("*.jpg")}
    orphan_labels += [p.name for p in (DFIRE_DIR / split / "labels").glob("*.txt")
                      if p.stem not in image_stems]

print(f"Images sans fichier de label : {(~df['has_label_file']).sum()}")
print(f"Labels sans image            : {len(orphan_labels)}")
print(f"Lignes de label invalides    : {df['n_invalid_lines'].sum()}")
print(f"Classes présentes            : {sorted(df_boxes['class_id'].unique())}")
print(f"Classes inconnues            : {(~df_boxes['class_id'].isin(CLASS_NAMES)).sum()}")

**Vérification de la convention de classes.** Le README officiel ne donne pas l'identifiant
numérique des classes, mais il donne le nombre de boîtes par classe (Fire 14 692, Smoke 11 865).
On compare avec les comptes réels pour confirmer `0 = smoke`, `1 = fire`.

In [ ]:
box_counts = df_boxes["class_id"].value_counts().sort_index()
readme_counts = {0: 11_865, 1: 14_692}   # Smoke, Fire d'après le README D-Fire
for cls, n in box_counts.items():
    status = "OK" if readme_counts.get(cls) == n else "ÉCART"
    print(f"classe {cls} ({CLASS_NAMES.get(cls)}) : {n} boîtes | README : {readme_counts.get(cls)} -> {status}")

**Valeurs aberrantes géométriques.**
- Coordonnées hors de [0, 1] : non conformes au format YOLO.
- Ultralytics (fonction `verify_image_label`, branche `main`) tolère jusqu'à 1,01 ; au-delà,
  l'image entière est **ignorée** comme « corrupt ». Ces cas doivent donc être corrigés (clipping)
  à l'étape de préparation plutôt que laissés tels quels.
- Boîtes d'aire nulle : inutilisables.

In [ ]:
coords = df_boxes[["x", "y", "w", "h"]]
out_of_range = df_boxes[((coords < 0) | (coords > 1)).any(axis=1)]
beyond_tol = df_boxes[(coords > 1.01).any(axis=1)]
zero_area = df_boxes[df_boxes["area"] <= 0]

print(f"Boîtes avec coordonnées hors [0, 1] : {len(out_of_range)}")
print(f"  dont > 1.01 (image ignorée par Ultralytics) : {len(beyond_tol)}")
print(f"Boîtes d'aire nulle : {len(zero_area)}")
out_of_range

## 6. Distribution des classes

Le projet traite deux tâches binaires : **fumée oui/non** et **feu oui/non**.
On regarde aussi les 4 catégories au niveau image.

In [ ]:
# Ordre et couleurs fixes des catégories (même couleur = même catégorie dans tout le notebook)
CATEGORY_ORDER = ["smoke_only", "fire_only", "fire_and_smoke", "none"]
CATEGORY_COLORS = {"smoke_only": "#2a78d6", "fire_only": "#eb6834",
                   "fire_and_smoke": "#1baf7a", "none": "#eda100"}
CLASS_COLORS = {"smoke": "#2a78d6", "fire": "#eb6834"}

# Effectifs et proportions par split
counts = pd.crosstab(df["category"], df["split"]).reindex(CATEGORY_ORDER)
props = pd.crosstab(df["category"], df["split"], normalize="columns").reindex(CATEGORY_ORDER)
display(counts.assign(total=counts.sum(axis=1)))
display((props * 100).round(1).rename(columns=lambda c: f"{c} (%)"))

# Prévalence des deux tâches binaires
display(df.groupby("split")[["has_smoke", "has_fire"]].mean().mul(100).round(1)
          .rename(columns={"has_smoke": "fumée (%)", "has_fire": "feu (%)"}))

In [ ]:
# Proportions par split : vérifie que train et test ont la même répartition
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(CATEGORY_ORDER))
width = 0.38
for i, split in enumerate(SPLITS):
    ax.bar(x + (i - 0.5) * width, props[split] * 100, width - 0.02,
           label=split, color=["#2a78d6", "#eb6834"][i])
ax.set_xticks(x, CATEGORY_ORDER)
ax.set_ylabel("% des images du split")
ax.set_title("Répartition des catégories par split")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Homogénéité des sources

Les images proviennent de trois sources (préfixe du nom de fichier) :
`AoF` (caméras de surveillance), `WEB` (images web), `PublicDataset` (jeux de données tiers).
Une source sur-représentée dans une catégorie peut créer un **biais** (le modèle apprend la source
plutôt que le feu).

In [ ]:
display(pd.crosstab(df["source"], df["split"], margins=True))
src_cat = pd.crosstab(df["source"], df["category"], normalize="index").reindex(columns=CATEGORY_ORDER)
display((src_cat * 100).round(1))

In [ ]:
# Barres empilées horizontales : composition de chaque source
fig, ax = plt.subplots(figsize=(8, 3))
left = np.zeros(len(src_cat))
for cat in CATEGORY_ORDER:
    vals = src_cat[cat].values * 100
    ax.barh(src_cat.index, vals, left=left, color=CATEGORY_COLORS[cat],
            label=cat, edgecolor="white", linewidth=2)
    left += vals
ax.set_xlabel("% des images de la source")
ax.set_title("Composition des catégories par source")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Statistiques des boîtes

La taille des objets conditionne la difficulté : une petite fumée lointaine (détection précoce)
occupe très peu de pixels.

In [ ]:
display(df_boxes.groupby("class_name")["area"].describe(percentiles=[0.1, 0.5, 0.9]).round(4))
display(df[df["n_boxes"] > 0]["n_boxes"].describe().round(2).to_frame("boîtes par image positive"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Aire relative des boîtes (échelle log) par classe
bins = np.logspace(-5, 0, 50)
for name in ["smoke", "fire"]:
    axes[0].hist(df_boxes.loc[df_boxes["class_name"] == name, "area"].clip(lower=1e-5),
                 bins=bins, alpha=0.6, label=name, color=CLASS_COLORS[name])
axes[0].set_xscale("log")
axes[0].set_xlabel("aire de la boîte / aire de l'image")
axes[0].set_ylabel("nombre de boîtes")
axes[0].set_title("Taille relative des objets")
axes[0].legend(frameon=False)

# Nombre de boîtes par image (images positives uniquement)
n_boxes_pos = df.loc[df["n_boxes"] > 0, "n_boxes"]
axes[1].hist(n_boxes_pos, bins=np.arange(1, n_boxes_pos.max() + 2) - 0.5, color="#2a78d6")
axes[1].set_xlabel("boîtes par image")
axes[1].set_ylabel("nombre d'images")
axes[1].set_title("Boîtes par image positive")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Vérification des images : lisibilité, résolution, couleur

Contrairement au P7 (échantillon de 200 images), **toutes** les images sont vérifiées.
Pour rester rapide, chaque JPEG est décodé en taille réduite (`Image.draft`), ce qui suffit à
détecter un fichier illisible et à calculer des statistiques globales.

On calcule aussi un **hash perceptuel (dHash)** : deux images visuellement identiques ont le même
hash. Cela permet de repérer des doublons, notamment entre train et test (fuite de données).

In [ ]:
def dhash(gray_img, size=8):
    """Hash perceptuel : compare chaque pixel à son voisin de droite (64 bits)."""
    small = np.asarray(gray_img.resize((size + 1, size)), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return "".join("1" if b else "0" for b in bits)


img_stats, corrupted = [], []
for row in df.itertuples():
    try:
        with Image.open(ROOT_DIR / row.path) as img:
            width, height, mode, fmt = *img.size, img.mode, img.format  # lu dans l'en-tête
            img.draft("RGB", (128, 128))                                 # décodage JPEG réduit
            rgb = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
            gray = img.convert("L")
        img_stats.append({
            "image_id": row.image_id, "width": width, "height": height,
            "mode": mode, "format": fmt,
            "brightness": rgb.mean(),          # luminosité moyenne [0, 1]
            "contrast": rgb.mean(axis=2).std(),  # écart-type des niveaux de gris
            "mean_r": rgb[..., 0].mean(), "mean_g": rgb[..., 1].mean(), "mean_b": rgb[..., 2].mean(),
            "dhash": dhash(gray),
        })
    except Exception as e:
        # Toute erreur PIL = image corrompue ou illisible
        corrupted.append({"path": row.path, "erreur": str(e)})

df = df.merge(pd.DataFrame(img_stats), on="image_id", how="left")
print(f"Images vérifiées : {len(img_stats)} | corrompues : {len(corrupted)}")
if corrupted:
    display(pd.DataFrame(corrupted))

In [ ]:
print("Formats :", df["format"].value_counts().to_dict())
print("Modes couleur :", df["mode"].value_counts().to_dict())
print(f"Résolutions distinctes : {df.groupby(['width', 'height']).ngroups}")
display(df.groupby(["width", "height"]).size().sort_values(ascending=False)
          .head(10).to_frame("nb images"))
display(df[["width", "height"]].describe().round(0))

In [ ]:
# Nuage largeur x hauteur : montre l'hétérogénéité des résolutions
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df["width"], df["height"], s=8, alpha=0.3, color="#2a78d6")
ax.set_xlabel("largeur (px)")
ax.set_ylabel("hauteur (px)")
ax.set_title("Résolution des images")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 10. Doublons visuels (risque de fuite train/test)

Les images `AoF` proviennent de caméras de surveillance : plusieurs peuvent être des images très
proches d'une même séquence. Ici on compte uniquement les **hash identiques** (distance 0) :
les quasi-doublons à hash légèrement différent ne sont pas comptés (sous-estimation), mais
deux images différentes peuvent ponctuellement partager un hash (collision) → chiffre indicatif.

In [ ]:
hash_splits = df.groupby("dhash")["split"].agg(set)
dup_groups = df.groupby("dhash").filter(lambda g: len(g) > 1)
cross = hash_splits[hash_splits.apply(lambda s: {"train", "test"} <= s)]
test_leak = df[(df["split"] == "test") & df["dhash"].isin(cross.index)]   # images de test concernées

print(f"Images partageant leur hash avec au moins une autre : {len(dup_groups)}")
print(f"Hash présents à la fois dans train et test          : {len(cross)}")
print(f"Images de test concernées                           : {len(test_leak)} "
      f"({len(test_leak) / (df['split'] == 'test').sum():.1%} du test)")

# Hash dégénérés (image uniforme : 64 bits identiques) = collisions probables
degenerate = {"0" * 64, "1" * 64}
print(f"Hash dégénérés parmi les hash communs               : {sum(h in degenerate for h in cross.index)}")

In [ ]:
# Profil des images de test concernées : source, catégorie, luminosité
display(test_leak["source"].value_counts().to_frame("images de test concernées"))
display(test_leak["category"].value_counts().reindex(CATEGORY_ORDER).to_frame("images de test concernées"))
print(f"Luminosité médiane : concernées {test_leak['brightness'].median():.3f} | "
      f"test global {df.loc[df['split'] == 'test', 'brightness'].median():.3f}")

# Cohérence des labels : un même hash porte-t-il la même catégorie ?
n_cat = df[df["dhash"].isin(cross.index)].groupby("dhash")["category"].nunique()
print(f"Hash communs avec une seule catégorie : {(n_cat == 1).sum()} / {len(n_cat)}")

In [ ]:
# Exemples tirés au hasard parmi les paires train/test au hash identique
examples = pd.Series(cross.index).sample(min(4, len(cross)), random_state=RANDOM_STATE)
if len(examples):
    fig, axes = plt.subplots(len(examples), 2, figsize=(7, 3 * len(examples)), squeeze=False)
    for i, h in enumerate(examples):
        for j, split in enumerate(["train", "test"]):
            row = df[(df["dhash"] == h) & (df["split"] == split)].iloc[0]
            axes[i, j].imshow(Image.open(ROOT_DIR / row["path"]))
            axes[i, j].set_title(f"{split} – {row['image_id']}", fontsize=9)
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()

## 11. Luminosité et couleur par catégorie et par source

Si la luminosité ou la couleur différait fortement selon la source, un modèle pourrait s'en servir
comme raccourci. Ces statistiques sont **exploratoires** : la normalisation appliquée au modèle
sera celle d'ImageNet (comme dans le P7).

In [ ]:
display(df.groupby("category")[["brightness", "contrast", "mean_r", "mean_g", "mean_b"]]
          .mean().reindex(CATEGORY_ORDER).round(3))
display(df.groupby("source")[["brightness", "contrast", "mean_r", "mean_g", "mean_b"]]
          .mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([df.loc[df["category"] == c, "brightness"] for c in CATEGORY_ORDER],
                showfliers=False)
axes[0].set_xticks(range(1, len(CATEGORY_ORDER) + 1), CATEGORY_ORDER)
axes[0].set_title("Luminosité par catégorie")
sources = sorted(df["source"].unique())
axes[1].boxplot([df.loc[df["source"] == s, "brightness"] for s in sources],
                showfliers=False)
axes[1].set_xticks(range(1, len(sources) + 1), sources)
axes[1].set_title("Luminosité par source")
for ax in axes:
    ax.set_ylabel("luminosité moyenne [0, 1]")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Visualisation d'exemples avec annotations

Une ligne par catégorie, 5 exemples tirés au hasard. Boîtes : bleu = fumée, orange = feu.

In [ ]:
def show_with_boxes(ax, row):
    """Affiche l'image et dessine ses boîtes YOLO (converties en pixels)."""
    img = Image.open(ROOT_DIR / row["path"])
    W, H = img.size
    ax.imshow(img)
    for b in df_boxes[df_boxes["image_id"] == row["image_id"]].itertuples():
        x0, y0 = (b.x - b.w / 2) * W, (b.y - b.h / 2) * H   # coin haut-gauche
        ax.add_patch(patches.Rectangle((x0, y0), b.w * W, b.h * H, fill=False,
                                       edgecolor=CLASS_COLORS[b.class_name], linewidth=2))
    ax.axis("off")


fig, axes = plt.subplots(len(CATEGORY_ORDER), 5, figsize=(16, 3.2 * len(CATEGORY_ORDER)))
fig.suptitle("Exemples par catégorie (bleu = fumée, orange = feu)", fontsize=14)
for i, cat in enumerate(CATEGORY_ORDER):
    sample = df[df["category"] == cat].sample(5, random_state=RANDOM_STATE)
    for j, (_, row) in enumerate(sample.iterrows()):
        show_with_boxes(axes[i, j], row)
        axes[i, j].set_title(f"{cat}\n{row['image_id']}", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Une ligne par source : aperçu de l'hétérogénéité visuelle
fig, axes = plt.subplots(len(sources), 5, figsize=(16, 3.2 * len(sources)))
fig.suptitle("Exemples par source", fontsize=14)
for i, src in enumerate(sources):
    sample = df[df["source"] == src].sample(5, random_state=RANDOM_STATE)
    for j, (_, row) in enumerate(sample.iterrows()):
        show_with_boxes(axes[i, j], row)
        axes[i, j].set_title(f"{src} – {row['category']}", fontsize=8)
plt.tight_layout()
plt.show()

## 13. Sauvegarde des métadonnées

In [ ]:
METADATA_PATH = PROCESSED_DIR / "metadata.csv"
BOXES_PATH = PROCESSED_DIR / "boxes.csv"

# Sauvegarde pour les notebooks suivants (chemins relatifs à la racine du projet)
df.to_csv(METADATA_PATH, index=False)
df_boxes.to_csv(BOXES_PATH, index=False)
print(f"Métadonnées sauvegardées : {METADATA_PATH.relative_to(ROOT_DIR)} – {df.shape}")
print(f"Boîtes sauvegardées      : {BOXES_PATH.relative_to(ROOT_DIR)} – {df_boxes.shape}")
print(f"Colonnes : {list(df.columns)}")

## 14. Observations et conclusions

_Chiffres issus de l'exécution du 13/11/2026 sur le dataset D-Fire complet._

**Volume et structure**
- **21 527 images** : 17 221 train / 4 306 test (découpage officiel), 26 557 boîtes.
- Aucune image sans label, aucun label orphelin, aucune ligne invalide, aucune classe inconnue.
- Convention confirmée : `0 = smoke` (11 865 boîtes), `1 = fire` (14 692 boîtes), identique au README.

**Distribution (deux tâches binaires)**
- Fumée présente : ~49 % des images ; feu présent : ~27 % → la tâche **feu** est déséquilibrée.
- Catégories : `none` 45,7 %, `smoke_only` 27,3 %, `fire_and_smoke` 21,6 %, `fire_only` 5,4 % (rare).
- Train et test ont des répartitions très proches (écarts ≤ 1,1 point) → split cohérent.

**Valeurs aberrantes à corriger en préparation**
- 8 boîtes « smoke » hors [0, 1] (toutes en test, images `WEB`), dont 4 > 1,01 → **clipping** à prévoir.
- 18 boîtes d'aire nulle (14 en train, 4 en test) → à supprimer.
- 54 fichiers `.jpg` sont en réalité des **PNG**, dont 48 en **RGBA** → conversion `RGB` obligatoire.
- Aucune image illisible.

**Biais potentiels**
- **Sources hétérogènes** : `AoF` est à 60 % `none`, `PublicDataset` à 67 % `smoke_only`, `WEB` concentre
  les `fire_and_smoke` (31 %). La source est donc corrélée au label → risque que le modèle apprenne la source.
- **Luminosité** : les images `fire_only` sont nettement plus sombres (moyenne 0,15 contre 0,28 pour
  `fire_and_smoke` et 0,38–0,44 pour `none` / `smoke_only`) → risque de raccourci « image sombre = feu ».
  Hypothèse à vérifier visuellement : beaucoup de ces images seraient prises de nuit.
- **Fuite train/test** : 687 hash perceptuels identiques entre train et test, soit **1 021 images de test
  (23,7 %)**, dont **905 `AoF`** (caméras de surveillance, images successives d'une même séquence).
  Ce n'est pas un artefact des images sombres : aucun hash dégénéré, et une luminosité médiane proche de
  celle du test global (0,391 contre 0,405). 640 des 687 hash portent la même catégorie en train et en test.
  Les exemples affichés confirment des images quasi identiques : ex. `AoF05821` (train) et `AoF08163` (test),
  même caméra, horodatages à 5 s d'écart. Une même image peut aussi apparaître sous deux sources
  (`PublicDataset00789` / `WEB11077`). Chiffre indicatif (hash identiques seulement), mais la fuite est
  réelle → à traiter avant toute évaluation finale.

**Images et objets**
- Résolutions très variées (3 366 distinctes ; 1280×720 la plus fréquente, 42,6 %) → redimensionnement
  224×224 pour ResNet50.
- Le **feu est un petit objet** : aire médiane 0,5 % de l'image (10 % des boîtes < 0,07 %), contre 14 % pour
  la fumée → une classification sur l'image entière réduite à 224×224 risque de manquer les petits feux.

**Suite (étape 2)** : conversion RGB, correction des boîtes, traitement des doublons, puis simulation
du faible étiquetage (sous-ensemble labellisé du train, le reste traité comme non labellisé).